In [12]:
import requests
from bs4 import BeautifulSoup
import time
import pandas as pd
import re
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

## Data Scrapping and Modification

In [63]:
city_urls = {
    "Hyderabad": "https://www.magicbricks.com/property-for-rent/residential-real-estate?cityName=Hyderabad",
    "Bangalore": "https://www.magicbricks.com/property-for-rent/residential-real-estate?cityName=Bangalore",
    "Mumbai": "https://www.magicbricks.com/property-for-rent/residential-real-estate?cityName=Mumbai",
    "Pune": "https://www.magicbricks.com/property-for-rent/residential-real-estate?cityName=Pune",
    "Chennai": "https://www.magicbricks.com/property-for-rent/residential-real-estate?cityName=Chennai"
}

# Define headers to prevent a browser from blocking
headers = {'User-Agent': 'Mozilla/5.0'}

max_properties_per_city = 800

# list to store all scraped data
all_data = []

# Keywords to detect actual property types from title text
property_keywords = ['Apartment', 'Flat', 'Studio', 'Villa', 'House', 'Floor', 'Penthouse', 'Row House', 'Farm']

# Loop through each city and its corresponding URL
for city, base_url in city_urls.items():
    print(f"\nScraping {city}...")
    city_data = []   # Store city-specific listings
    page = 1      # Start from page 1

    # Loop through pages until desired property count is reached
    while len(city_data) < max_properties_per_city:
        url = f"{base_url}&page={page}"
        res = requests.get(url, headers=headers)
        soup = BeautifulSoup(res.text, 'html.parser')

        # Find all property listing cards on the page
        listings = soup.find_all("div", class_="mb-srp__card")
        if not listings:
            print(" No listings found, stopping.")
            break

        # Loop through each property card
        for listing in listings:
            try:
                # Extract bhk from title
                title_tag = listing.find("h2", class_="mb-srp__card--title")
                title = title_tag.get_text(strip=True)

                bhk_match = re.search(r"(\d+)\s*BHK", title.upper())
                bhk = bhk_match.group(1) if bhk_match else None

                # Extract location
                locality = title.split("in", 1)[-1].strip() if "in" in title else None

                # Detect property type from title
                title_clean = title.replace(",", " ").replace("-", " ")
                title_words = title_clean.split()
                prop_type_from_title = next((word for word in title_words if word.capitalize() in property_keywords), None)
            except:
                bhk = locality = prop_type_from_title = None

            # Extract rent price
            try:
                rent = listing.find("div", class_="mb-srp__card__price--amount").get_text(strip=True)
                rent = rent.replace("₹", "").replace(",", "").strip()
            except:
                rent = None

            summary = listing.find("div", class_="mb-srp__card__summary__list")
            summary_items = summary.find_all("div", class_="mb-srp__card__summary__list--item") if summary else []

            # Initializing Variable
            area = furnishing = facing = property_type = overlooking = Bathroom = Balcony=TenantPreferred=Availability= None
            
            # Loop through summary items and extract specific features
            for item in summary_items:
                try:
                    label = item.find("div", class_="mb-srp__card__summary--label").get_text(strip=True).lower()
                    value = item.find("div", class_="mb-srp__card__summary--value").get_text(strip=True)

                    if "area" in label:
                        area = value
                    elif "furnish" in label:
                        furnishing = value
                    elif "facing" in label:
                        facing = value
                    elif "property type" in label or "type" in label:
                        property_type = value
                    elif "overlooking" in label :
                        overlooking = value
                    elif "bathroom" in label:
                        Bathroom = value  
                    elif "balcony" in label:
                        Balcony = value 
                    elif "tenant preferred" in label:
                        TenantPreferred=value
                    elif "availability" in label:
                        Availability=value
                        

                except:
                    continue

            # Final fallback if property_type is missing or invalid
            if not property_type or property_type.lower() in ["for", "in", "rent", "sale"]:
                property_type = prop_type_from_title

            if not bhk or not locality:
                continue

            city_data.append({
                "City": city,
                "BHK": bhk.strip(),
                "Location": locality.strip(),
                "Price (₹)": rent,
                "Area (sqft)": area.strip() if area else None,
                "Property Type": property_type.strip() if property_type else None,
                "Furnishing": furnishing.strip() if furnishing else None,
                "Property Facing": facing.strip() if facing else None,
                "overlooking":overlooking.strip() if overlooking else None,
                "Bathroom":Bathroom.strip() if Bathroom else None,
                "Balcony":Balcony.strip() if Balcony else None,
                "Tenant Preferred":TenantPreferred.strip() if TenantPreferred else None,
                "Availability":Availability.strip() if Availability else None,

                
            })

            if len(city_data) >= max_properties_per_city:
                break

        page += 1    # Move to next Page
        time.sleep(1)  # delay to avoid overloading the server


    # Append city data to the master list
    all_data.extend(city_data)
print("\nScraping completed...")


Scraping Hyderabad...

Scraping Bangalore...

Scraping Mumbai...

Scraping Pune...

Scraping Chennai...

Scraping completed...


## Data Handling

In [42]:
# Save to CSV
df = pd.DataFrame(all_data)
df.to_csv("MagicBricksProject_Scraped_Data.csv", index=False)
print("\nData saved to MagicBricksProject_Scraped_Data.csv")


Data saved to MagicBricksProject_Scraped_Data.csv


In [43]:
# DataFrame overview
df.head(5)

,City,BHK,Location,Price (₹),Area (sqft),Property Type,Furnishing,Property Facing,overlooking,Bathroom,Balcony,Tenant Preferred,Availability
0,Hyderabad,4,"Kokapet, Outer Ring Road Hyderabad",2.3 Lac,5030 sqft,Villa,Semi-Furnished,East,Garden/Park,5,1,Bachelors/Family,Immediately
1,Hyderabad,3,"Aparna Serene Park, Kondapur, Hyderabad",68000,1500 sqft,Flat,Semi-Furnished,West,Main Road,3,1,Bachelors/Family,Immediately
2,Hyderabad,3,"Prestige Tranquil, Kokapet, Outer Ring Road, H...",75000,1361 sqft,Flat,Semi-Furnished,East,"Pool, Garden/Park, Main Road",3,2,None,None
3,Hyderabad,1,"Kondapur, Hyderabad",15500,800 sqft,Flat,Semi-Furnished,North,Main Road,1,1,Bachelors,Immediately
4,Hyderabad,3,"Creative RVR Udaya Creative, Kondapur, Hyderabad",55000,2000 sqft,Flat,Semi-Furnished,East,Main Road,3,3,Bachelors/Family,Immediately


In [44]:
# Number of Columns
print("Number of columns:",df.shape[1])

Number of columns: 13


In [45]:
# Number of  Rows

print("Number of rows:",df.shape[0])

Number of rows: 4000


In [46]:
# Print the data types of each column
print("Data types of each column:\n",df.dtypes)

Data types of each column:
 City                object
BHK                 object
Location            object
Price (₹)           object
Area (sqft)         object
Property Type       object
Furnishing          object
Property Facing     object
overlooking         object
Bathroom            object
Balcony             object
Tenant Preferred    object
Availability        object
dtype: object


In [47]:
# Print the number of missing values in each column
print("Missing values in each column:\n",df.isnull().sum())

Missing values in each column:
 City                   0
BHK                    0
Location               0
Price (₹)              0
Area (sqft)            0
Property Type          0
Furnishing            15
Property Facing      918
overlooking         1254
Bathroom              10
Balcony             1075
Tenant Preferred       4
Availability           4
dtype: int64


In [49]:
# It shows the duplicated rows count
num_duplicates = df.duplicated().sum()
print(f"Total duplicate rows (all columns): {num_duplicates}")

Total duplicate rows (all columns): 2


In [50]:
#Duplicate data
df[df.duplicated]

,City,BHK,Location,Price (₹),Area (sqft),Property Type,Furnishing,Property Facing,overlooking,Bathroom,Balcony,Tenant Preferred,Availability
3441,Chennai,2,"Bhaggyam Griha, Thoraipakkam, Chennai",35000,1452 sqft,Flat,Semi-Furnished,None,None,2,None,Bachelors/Family,Immediately
3944,Chennai,3,Kodungaiyur Chennai,22000,1200 sqft,House,Unfurnished,West,None,2,1,Bachelors,Immediately


In [ ]:
#drop duplicates
df.drop_duplicates(inplace=True)

In [52]:
# Drop duplicate
num_duplicates = df.duplicated().sum()
print(f"Total duplicate rows (all columns): {num_duplicates}")

Total duplicate rows (all columns): 0


In [53]:
print("Number of rows:",df.shape[0])

Number of rows: 3998


## Data Cleaning

In [56]:
df.isna().sum()

City                   0
BHK                    0
Location            2106
Price (₹)              0
Area (sqft)            0
Property Type          0
Furnishing            15
Property Facing      917
overlooking         1252
Bathroom              10
Balcony             1074
Tenant Preferred       4
Availability           4
dtype: int64

In [ ]:
# # List of locations 
# localities = [
#     "Kondapur", "Narsingi", "Gachibowli", "Tellapur", "Kompally", "Kollur", "Shamshabad","Nalagandla",
#     "Bachupally", "Miyapur", "Hitech City", "Puppalaguda", "Kokapet", "Manikonda", "Madhapur",
#     "Whitefield", "Varthur", "Hebbal", "Yelahanka", "Devanahalli",
#     "Sarjapur", "Electronic City", "Kanakapura Road", "Bannerghatta Main Road","Rajajinagar",
#     "Mulund", "Andheri", "Chembur", "Worli", "Powai", "Borivali", "Bandra", "Wadala",
#     "Goregaon", "Kandivali East", "Thakur Village", "Juhu", "Kandivali", "Hadapsar", "Mundhwa",
#     "Kharadi", "Baner", "Hadapsar", "Hinjawadi", "Wagholi", "Wakad", "Balewadi",
#     "NIBM Road", "Viman Nagar", "Undri", "Sholinganallur", "Medavakkam", "Pallikaranai", "Porur", "Perungudi", 
#     "Pallavaram", "Guduvancheri", "Kelambakkam", "Thiruvanmiyur", "Padur"
    
# ]

# # Lowercased for comparison
# localities_lower = [loc.lower() for loc in localities]

# # Function to clean location
# def clean_location(text):
#     if pd.isna(text):
#         return ''
#     text = re.sub(r'\s+', ' ', text).lower()
#     for loc, loc_lower in zip(localities, localities_lower):
#         if loc_lower in text:
#             return loc
#     return np.nan

# # Update Location column in-place
# df['Location'] = df['Location'].apply(clean_location)  

In [58]:
# List of locations 
localities = [
    "Kondapur", "Narsingi", "Gachibowli", "Tellapur", "Kompally", "Kollur", "Shamshabad","Nalagandla",
    "Bachupally", "Miyapur", "Hitech City", "Puppalaguda", "Kokapet", "Manikonda", "Madhapur",
    "Whitefield", "Varthur", "Hebbal", "Yelahanka", "Devanahalli",
    "Sarjapur", "Electronic City", "Kanakapura Road", "Bannerghatta Main Road","Rajajinagar",
    "Mulund", "Andheri", "Chembur", "Worli", "Powai", "Borivali", "Bandra", "Wadala",
    "Goregaon", "Kandivali East", "Thakur Village", "Juhu", "Kandivali", "Hadapsar", "Mundhwa",
    "Kharadi", "Baner", "Hadapsar", "Hinjawadi", "Wagholi", "Wakad", "Balewadi",
    "NIBM Road", "Viman Nagar", "Undri", "Sholinganallur", "Medavakkam", "Pallikaranai", "Porur", "Perungudi", 
    "Pallavaram", "Guduvancheri", "Kelambakkam", "Thiruvanmiyur", "Padur"
    # ---------------- PUNE ----------------
    "Shivaji Nagar", "Deccan Gymkhana", "Camp", "Swargate", "Kothrud", "Aundh",
    "Baner", "Bavdhan", "Pashan", "Warje", "Viman Nagar", "Koregaon Park",
    "Kalyani Nagar", "Kharadi", "Hadapsar", "Wagholi", "Mundhwa", "Bibwewadi",
    "Katraj", "Dhayari", "Hinjewadi", "Wakad", "Pimple Saudagar", "Pimpri",

    # ---------------- HYDERABAD ----------------
    "Charminar", "Malakpet", "Nampally", "Asif Nagar", "Gachibowli",
    "HITEC City", "Madhapur", "Kondapur", "Financial District", "Nanakramguda",
    "Kukatpally", "Miyapur", "Chanda Nagar", "Hafeezpet", "Bachupally",
    "Nizampet", "Manikonda", "Narsingi", "Uppal", "Begumpet",
    "Banjara Hills", "Jubilee Hills", "Punjagutta", "Secunderabad",

    # ---------------- MUMBAI ----------------
    "Andheri", "Andheri East", "Andheri West", "Bandra", "Bandra East", "Bandra West",
    "Borivali", "Borivali East", "Borivali West", "Dadar", "Dadar East", "Dadar West",
    "Goregaon", "Goregaon East", "Goregaon West", "Juhu", "Kandivali",
    "Kandivali East", "Kandivali West", "Malad", "Malad East", "Malad West",
    "Lower Parel", "Worli", "Powai", "Chembur", "Ghatkopar", "Kurla",
    "Thane", "Navi Mumbai", "Vashi", "Nerul", "Kharghar",

    # ---------------- CHENNAI ----------------
    "T Nagar", "Adyar", "Velachery", "Anna Nagar", "Tambaram", "Chromepet",
    "Guindy", "Porur", "Vadapalani", "Nungambakkam", "Mylapore", "Kodambakkam",
    "Royapettah", "Egmore", "Triplicane", "Thiruvanmiyur", "Perungudi",
    "Sholinganallur", "OMR", "ECR", "Pallavaram", "Medavakkam",

    # ---------------- BANGALORE ----------------
    "Whitefield", "Marathahalli", "KR Puram", "Indiranagar", "Koramangala",
    "HSR Layout", "BTM Layout", "Electronic City", "Bellandur", "Sarjapur Road",
    "MG Road", "Jayanagar", "JP Nagar", "Banashankari", "Yelahanka",
    "Hebbal", "Malleshwaram", "Rajajinagar", "Basavanagudi", "Kengeri",
    "Nagarbhavi", "Vijayanagar"
]

# Lowercased for comparison
localities_lower = [loc.lower() for loc in localities]

# Function to clean location
def clean_location(text):
    if pd.isna(text):
        return ''
    text = re.sub(r'\s+', ' ', text).lower()
    for loc, loc_lower in zip(localities, localities_lower):
        if loc_lower in text:
            return loc
    return np.nan

# Update Location column in-place
df['Location'] = df['Location'].apply(clean_location)  

In [38]:
df=df.dropna().reset_index(drop=True)

In [59]:
df

,City,BHK,Location,Price (₹),Area (sqft),Property Type,Furnishing,Property Facing,overlooking,Bathroom,Balcony,Tenant Preferred,Availability
0,Hyderabad,4,Kokapet,2.3 Lac,5030 sqft,Villa,Semi-Furnished,East,Garden/Park,5,1,Bachelors/Family,Immediately
1,Hyderabad,3,Kondapur,68000,1500 sqft,Flat,Semi-Furnished,West,Main Road,3,1,Bachelors/Family,Immediately
2,Hyderabad,3,Kokapet,75000,1361 sqft,Flat,Semi-Furnished,East,"Pool, Garden/Park, Main Road",3,2,None,None
3,Hyderabad,1,Kondapur,15500,800 sqft,Flat,Semi-Furnished,North,Main Road,1,1,Bachelors,Immediately
4,Hyderabad,3,Kondapur,55000,2000 sqft,Flat,Semi-Furnished,East,Main Road,3,3,Bachelors/Family,Immediately
...,...,...,...,...,...,...,...,...,...,...,...,...,...
3995,Chennai,2,Perungudi,60000,1187 sqft,Flat,Furnished,West,Garden/Park,2,1,Bachelors,Immediately
3996,Chennai,2,,20000,810 sqft,House,Semi-Furnished,None,None,2,None,Bachelors/Family,Immediately
3997,Chennai,2,,23000,1500 sqft,House,Semi-Furnished,None,None,3,1,Bachelors,Immediately
3998,Chennai,2,,15000,1800 sqft,House,Unfurnished,East,None,2,None,Bachelors/Family,Immediately


In [62]:
df["Location"].unique()

array(['Kokapet', 'Kondapur', 'Gachibowli', '', 'Nalagandla',
       'Secunderabad', 'Narsingi', 'Manikonda', 'Hitech City', 'Miyapur',
       'Banjara Hills', 'Chanda Nagar', 'Nizampet', 'Uppal',
       'Financial District', 'Hafeezpet', 'Kompally', 'Tellapur',
       'Jubilee Hills', 'Asif Nagar', 'Kollur', 'Kukatpally',
       'Bachupally', 'Whitefield', 'Madhapur', 'Punjagutta', 'Begumpet',
       'T Nagar', 'Shamshabad', 'Malakpet', 'Nampally', 'Rajajinagar',
       'Kanakapura Road', 'Bellandur', 'Marathahalli',
       'Bannerghatta Main Road', 'BTM Layout', 'Kengeri', 'Varthur',
       'Sarjapur', 'Jayanagar', 'Hebbal', 'Banashankari', 'Yelahanka',
       'Basavanagudi', 'Indiranagar', 'HSR Layout', 'JP Nagar',
       'Electronic City', 'Koramangala', 'Devanahalli', 'Nagarbhavi',
       'Powai', 'Wadala', 'Worli', 'Juhu', 'Goregaon', 'Andheri',
       'Bandra', 'Chembur', 'Malad', 'Kandivali East', 'Mulund',
       'Ghatkopar', 'Kurla', 'Lower Parel', 'Borivali', 'Dadar',
      

In [241]:
#Fixing Nan value with Empty
df["Location"]=df["Location"].fillna(" ")

In [57]:
print(df["Furnishing"].unique())
df[df["Furnishing"].isna()]

['Semi-Furnished' 'Unfurnished' 'Furnished' None]


,City,BHK,Location,Price (₹),Area (sqft),Property Type,Furnishing,Property Facing,overlooking,Bathroom,Balcony,Tenant Preferred,Availability
51,Hyderabad,3,Chanda Nagar,32000,1550 sqft,Apartment,None,None,None,2,2,Family,Immediately
522,Hyderabad,4,NaN,45000,4000 sqft,House,None,None,None,None,None,Bachelors/Family,Immediately
526,Hyderabad,4,NaN,40000,2418 sqft,Villa,None,None,None,3,2,Bachelors/Family,Immediately
1263,Bangalore,2,NaN,28000,1220 sqft,Flat,None,None,None,None,None,Bachelors/Family,Immediately
1395,Bangalore,3,NaN,75000,2300 sqft,Flat,None,None,None,None,None,Bachelors/Family,Immediately
1448,Bangalore,1,Yelahanka,26000,800 sqft,Apartment,None,None,None,1,None,Bachelors/Family,From Feb '26
1518,Bangalore,1,NaN,18500,600 sqft,Apartment,None,None,None,1,None,Bachelors,Immediately
1590,Bangalore,2,NaN,7 Lac,1100 sqft,Apartment,None,None,None,None,None,Bachelors/Family,Immediately
3183,Pune,2,NaN,12000,550 sqft,Flat,None,None,None,None,None,Bachelors/Family,Immediately
3259,Chennai,3,Perungudi,1.2 Lac,1300 sqft,Apartment,None,None,None,3,3,Bachelors/Family,Immediately


In [ ]:
df.groupby(by=["Furnishing"])["Furnishing"].agg(["count"])

,count
Furnishing,
Furnished,335
Semi-Furnished,854
Unfurnished,292


In [253]:
df["Furnishing"].mode()[0]

'Semi-Furnished'

In above Group-by table we can see that, Semi-Furnished	has more count then Furnished and Unfurnished. I have decieded to assign not available values with maximum repeating value,i.e is "Semi-Furnished".

In [245]:
#fixed with "Semi-Furnished"
df["Furnishing"]=df["Furnishing"].fillna("Semi-Furnished")

In [260]:
df.groupby("Property Facing")["Property Facing"].value_counts()

Property Facing
East            757
North           171
North - East    103
North - West     24
South            40
South - East      9
South -West       9
West            151
Name: count, dtype: int64

In [267]:
df["Property Facing"].mode()[0]

'East'

In [264]:
df["Property Facing"]=df["Property Facing"].fillna(df["Property Facing"].mode()[0])

In [270]:
df["overlooking"].unique()

array(['Pool, Garden/Park, Main Road', None,
       'Garden/Park, Pool, Main Road', 'Garden/Park',
       'Garden/Park, Main Road', 'Main Road', 'Pool', 'Garden/Park, Pool',
       'Pool, Main Road', 'Main Road, Garden/Park, Pool',
       'Main Road, Garden/Park', 'Pool, Garden/Park',
       'Pool, Main Road, Garden/Park', 'Garden/Park, Main Road, Pool'],
      dtype=object)

In [273]:
df["overlooking"].isna().sum()

np.int64(316)

In [ ]:
df.groupby("overlooking")["overlooking"].value_counts()


overlooking
Garden/Park                     299
Garden/Park, Main Road          189
Garden/Park, Main Road, Pool      1
Garden/Park, Pool                56
Garden/Park, Pool, Main Road    152
Main Road                       340
Main Road, Garden/Park           13
Main Road, Garden/Park, Pool     14
Pool                             18
Pool, Garden/Park                18
Pool, Garden/Park, Main Road     58
Pool, Main Road                   8
Pool, Main Road, Garden/Park      1
Name: count, dtype: int64

In [276]:
print(df["overlooking"].mode()[0])
df["overlooking"]=df["overlooking"].fillna(df["overlooking"].mode()[0])

Main Road


In [333]:
df["Bathroom"]=df["Bathroom"].fillna(df["Bathroom"].mode()[0])

In [337]:
df.groupby("Balcony")["Balcony"].value_counts()

Balcony
1     462
10      1
2     478
3     164
4      43
5       4
6       1
Name: count, dtype: int64

In [64]:
df["Balcony"]=df["Balcony"].fillna("Unknown")

In [65]:

df.groupby("Tenant Preferred")["Tenant Preferred"].value_counts()

Tenant Preferred
Bachelors           1334
Bachelors/Family    2077
Family               583
Name: count, dtype: int64

In [366]:
df["Tenant Preferred"]=df["Tenant Preferred"].fillna(df["Tenant Preferred"].mode()[0])

In [370]:
df["Availability"]=df["Availability"].fillna(df["Availability"].mode()[0])

In [4]:
df=pd.read_csv("Cleaned_Data.csv")

In [5]:
df.drop(columns="Unnamed: 0",inplace=True)


In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1483 entries, 0 to 1482
Data columns (total 13 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   City              1483 non-null   object
 1   BHK               1483 non-null   int64 
 2   Location          1483 non-null   object
 3   Price (₹)         1483 non-null   object
 4   Area (sqft)       1483 non-null   object
 5   Property Type     1483 non-null   object
 6   Furnishing        1483 non-null   object
 7   Property Facing   1483 non-null   object
 8   overlooking       1483 non-null   object
 9   Bathroom          1483 non-null   int64 
 10  Balcony           1483 non-null   object
 11  Tenant Preferred  1483 non-null   object
 12  Availability      1483 non-null   object
dtypes: int64(2), object(11)
memory usage: 150.7+ KB


In [7]:
df.head(1)

,City,BHK,Location,Price (₹),Area (sqft),Property Type,Furnishing,Property Facing,overlooking,Bathroom,Balcony,Tenant Preferred,Availability
0,Hyderabad,3,Kokapet,85000,1335 sqft,Flat,Semi-Furnished,East,"Pool, Garden/Park, Main Road",3,2,Bachelors/Family,Immediately


## Outliers

In [8]:
len(df) - 706

777

In [9]:
df.groupby("Location").size().reset_index(name="count")

,Location,count
0,,706
1,Adyar,4
2,Andheri,23
3,Anna Nagar,8
4,Asif Nagar,1
...,...,...
103,Wakad,11
104,Warje,2
105,Whitefield,40
106,Worli,15
